## Ricavo funzione


In [1]:
import ROOT
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali

In [2]:
def Sorgente0(x, par):                  # nessuna modulazione
    t = x[0]
    return par[0]*(1 + par[1]*t)

def Sorgente(x, par):                   # sorgente parzialmente polarizzata
    t = x[0]
    return par[0]*(1 + par[1]*t)*(1 + par[2]*np.cos(2*t - par[3]))

def modellopy_S(t, A, D, E, F):
    return A*(1 + D*t)*(1 + E*np.cos(2*t - F))

def modellopy_S0(t, A, D):
    return A*(1 + D*t)

def deriv_S(x, par):
    dt = 1e-6
    return (modellopy_S(x+dt/2, par[0], par[1], par[2], par[3])
          - modellopy_S(x-dt/2, par[0], par[1], par[2], par[3]))/dt

def deriv_S0(x, par):
    dt = 1e-6
    return (modellopy_S0(x+dt/2, par[0], par[1]) - modellopy_S0(x-dt/2, par[0], par[1]))/dt


In [3]:
xlsx ="Esperienza4.xlsx"
df = pd.read_excel(xlsx, sheet_name= "Dati")
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.to_numpy(dtype = 'float64')

x = df[:, 1]     
y = df[:, 2]     

err_x = df[:, 3]     
err_y = df[:, 4]   

# eliminazione primi dati
index = [i for i in range(6)]

x = np.delete(x, index)
y = np.delete(y, index)
err_x = np.delete(err_x, index)
err_y = np.delete(err_y, index)

In [4]:
gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Solo sorgente + P1; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20); gr1.SetMarkerSize(0.7)
gr1.SetMarkerColor(ROOT.kAzure + 2); gr1.SetLineColor(ROOT.kAzure + 3)

# --- fit 0: nessuna modulazione ---
f0 = ROOT.TF1('f0', Sorgente0, min(x), max(x), 2)
f0.SetParameters(y.mean(), 0)
f0.SetParNames("A", "D")
r0 = gr1.Fit(f0, "S0")
chi2_0, ndf_0 = r0.Chi2(), r0.Ndf()

# --- fit 1: con modulazione; E e F iniziali dalla prima armonica a 2θ ---
yc = y - y.mean()
a = 2*np.mean(yc*np.cos(2*x));  b = 2*np.mean(yc*np.sin(2*x))
E0, F0 = np.hypot(a, b)/y.mean(), np.arctan2(b, a)

f1 = ROOT.TF1('f1', Sorgente, min(x), max(x), 4)
f1.SetParameters(y.mean(), 0, E0, F0)
f1.SetParNames("A", "D", "E", "F")
r1 = gr1.Fit(f1, "S")
chi2_1, ndf_1 = r1.Chi2(), r1.Ndf()

par_S = f1.GetParameters(); err_S = f1.GetParErrors()
for i, n in enumerate(["A", "D", "E", "F"]):
    print(f"{n} = {par_S[i]} ± {err_S[i]}")
print(f"chi2_0/ndf = {chi2_0:.1f}/{ndf_0},  chi2_1/ndf = {chi2_1:.1f}/{ndf_1}")

f_stat = ((chi2_0 - chi2_1)/(ndf_0 - ndf_1)) / (chi2_1/ndf_1)
print(f"F = {f_stat:.1f},  p = {f_dist.sf(f_stat, ndf_0 - ndf_1, ndf_1):.2e}")

err_yeff = np.sqrt(err_y**2 + (deriv_S(x, par_S)*err_x)**2)

c1 = ROOT.TCanvas("c1", "Fit sorgente", 800, 600); c1.SetGrid()
gr1.Draw("AP"); f1.SetLineColor(ROOT.kRed); f1.Draw("same")
c1.SaveAs("Immagini/sorgente_fit.png")


****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       658064
NDf                       =          192
Edm                       =   7.6237e-07
NCalls                    =           54
A                         =      3.52412   +/-   0.00180012  
D                         =    0.0242632   +/-   0.000137712 
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      651.702
NDf                       =          190
Edm                       =  8.07263e-07
NCalls                    =           97
A                         =      3.79797   +/-   0.00258466  
D                         =  -0.00159749   +/-   0.000159404 
E                         =     0.262503   +/-   0.00041609  
F                         =   -0.0434072   +/-   0.00206587  
A = 3.797966345876861 ± 0.0025846573230094852
D = -0.001597490583371432 ± 0.0001594042066286174
E = 0.26250341058875426 ± 0.00041608986165415896
F = -0.043407

Info in <TCanvas::Print>: png file Immagini/sorgente_fit.png has been created


In [5]:
y_attesi = np.array([f1.Eval(v) for v in x])
residuals_norm = (y - y_attesi)/err_yeff
residuals = y - y_attesi


In [6]:
E_old, sE_old = 0.0673, 0.0024
F_old, sF_old = 2.775, 0.035
dF = (par_S[3] - F_old + np.pi) % (2*np.pi) - np.pi      # differenza di fase riportata in (-π, π]
for nome, d, s in (("E", par_S[2]-E_old, np.hypot(err_S[2], sE_old)),
                   ("F", dF,             np.hypot(err_S[3], sF_old))):
    print(f"{nome}: differenza = {d:.4f} ± {s:.4f}  ({abs(d)/s:.1f} σ)")


E: differenza = 0.1952 ± 0.0024  (80.1 σ)
F: differenza = -2.8184 ± 0.0351  (80.4 σ)
